[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CONNECTS-SCV/bio-guides/blob/main/boltzgen/advanced/02_input_data_prep/02_data_prep.ipynb)


# 02 — 입력 데이터 준비

> 본문 [`02_input_data_prep.md`](02_input_data_prep.md) 와 **한 절씩 짝지어** 보세요.


## 0) 준비 — 저장소 클론 & 작업 폴더 이동

이 셀이 저장소를 클론하고 `02_input_data_prep/` 로 이동합니다. 로컬에서 `02_input_data_prep/` 안에 열었다면 클론 없이 진행돼요.

In [ ]:
REPO_URL = "https://github.com/CONNECTS-SCV/bio-guides.git"   # fork 했다면 본인 주소로 바꾸세요
CLONE_AS = "bio-guides"
CHAPTER  = "02_input_data_prep"
PIP_PKGS = "gemmi"   # 없으면 설치할 분석 라이브러리

import os, sys, subprocess, pathlib
IN_COLAB = "google.colab" in sys.modules

# HF 가중치 다운로드가 멈춘 채 매달리지 않도록 타임아웃을 겁니다.
os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "30")   # 스트림 30초 무응답 → 끊고 재시도
os.environ.setdefault("HF_HUB_ETAG_TIMEOUT", "15")

def _run(cmd, quiet=False):
    """quiet=True 면 출력을 삼키고 **실패했을 때만** 보여줘요.
    apt-get 은 "(Reading database ... 5%(Reading database ... 10%" 같은 진행률을 600자 넘게 쏟아내는데,
    그게 노트북을 연 학습자가 보는 첫 화면을 덮어버려서 실패로 오해하게 만들거든요."""
    print("$", cmd)
    if not quiet:
        subprocess.run(cmd, shell=True, check=True); return
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if p.returncode != 0:
        print((p.stdout or "") + (p.stderr or ""))
        raise subprocess.CalledProcessError(p.returncode, cmd)

_MARK = "boltzgen_viz.py"          # 이 파일이 있는 폴더가 advanced/ 루트

def _find_root():
    """advanced/ 루트를 찾습니다."""
    cwd = pathlib.Path.cwd()
    for base in (cwd, *list(cwd.parents)[:3]):
        if (base / _MARK).exists():
            return base
    for pat in (f"*/{_MARK}", f"*/*/{_MARK}", f"*/*/*/{_MARK}"):   # 클론 직후: cwd 아래로만 깊이 3까지
        hits = sorted(cwd.glob(pat))
        if hits:
            return hits[0].parent
    return None

ROOT = _find_root()
if ROOT is None and IN_COLAB:
    if not pathlib.Path(CLONE_AS).exists():
        _run(f'git clone --depth 1 "{REPO_URL}" {CLONE_AS}')
    ROOT = _find_root()
assert ROOT is not None, "advanced/ 루트를 못 찾았습니다. 로컬이면 이 노트북을 챕터 폴더 안에서 여세요."

ADV_ROOT = ROOT.resolve()
os.chdir(ADV_ROOT / CHAPTER)          # 이 챕터 폴더로 이동 → data/ 상대경로가 그대로 동작
sys.path.insert(0, str(ADV_ROOT))     # boltzgen_viz import 보장
import glob as _glob
if IN_COLAB and not _glob.glob("/usr/share/fonts/**/*Nanum*", recursive=True):
    # Colab 에는 한글 폰트가 없어 그래프의 한글 라벨이 □ 로 깨집니다.
    _run("apt-get -qq update", quiet=True)
    _run("DEBIAN_FRONTEND=noninteractive apt-get -qq install -y fonts-nanum", quiet=True)

# import 안 되는 패키지만 설치합니다.
import importlib, importlib.util
_IMPORT_NAME = {"scikit-learn": "sklearn", "pillow": "PIL", "biopython": "Bio"}
def _have(mod):
    try: return importlib.util.find_spec(mod) is not None
    except Exception: return False
def _ensure(pkgs):
    miss = [p for p in pkgs.split() if not _have(_IMPORT_NAME.get(p, p))]
    if miss:
        print("필요 라이브러리 설치:", " ".join(miss))
        _run(f'"{sys.executable}" -m pip -q install ' + " ".join(miss))  # python -m pip (Ch.03 권고)
        importlib.invalidate_caches()
if PIP_PKGS:
    _ensure(PIP_PKGS)

# 내 결과는 my_run/ 에 쌓이고, 없으면 커밋된 레퍼런스로 폴백합니다.
MY  = pathlib.Path("my_run")
MY.mkdir(exist_ok=True)

def find_one(pattern, ref, quiet=False):
    """my_run/ 에서 먼저 찾고, 없으면 레퍼런스 폴더에서 찾습니다."""
    for base in (MY/"final_ranked_designs", MY/"intermediate_designs_inverse_folded", MY):
        hits = sorted(base.glob(pattern))
        if hits:
            if not quiet: print(f"[내 결과]   {hits[0]}")
            return hits[0]
    hits = sorted(pathlib.Path(ref).glob(pattern))
    assert hits, f"{ref} 에서 '{pattern}' 을 찾지 못했습니다."
    if not quiet: print(f"[레퍼런스] {hits[0]}")
    return hits[0]

def cols_in(df, *names):
    """내 실행 결과와 레퍼런스는 컬럼 구성이 조금 다를 수 있어, 있는 컬럼만 고른다."""
    missing = [c for c in names if c not in df.columns]
    if missing:
        print("(이 실행에는 없는 컬럼 — 건너뜁니다:", ", ".join(missing) + ")")
    return [c for c in names if c in df.columns]

def inherit_run(*rel_paths):
    """앞 챕터에서 돌린 설계 결과를 이어받습니다(없으면 레퍼런스로 폴백)."""
    global MY
    if (MY / "final_ranked_designs").exists():
        return MY
    for rel in rel_paths:
        p = pathlib.Path(rel)
        if (p / "final_ranked_designs").exists():
            print("[이어받기] 앞 챕터에서 직접 돌린 결과를 사용합니다:", p)
            MY = p
            return MY
    return MY

# 레퍼런스 그림을 덮어쓰지 않도록 my_ 접두어.
def my_fig(name):
    return f"my_{name}"

print("작업 폴더 :", pathlib.Path.cwd())

## 1) 타깃 구조 받기 (본문 2.3)

설계는 "무엇에 붙일지"부터 정해집니다. 면역항암 표적 PD-L1 의 결정구조 `7uxq` 를 RCSB 에서 받아
이 챕터 내내 같은 재료로 씁니다.

In [ ]:
import urllib.request, pathlib
work = pathlib.Path("work"); work.mkdir(exist_ok=True)
PDB_ID = "7uxq"                                # PD-L1
cif = work / f"{PDB_ID}.cif"
if not cif.exists():
    urllib.request.urlretrieve(f"https://files.rcsb.org/download/{PDB_ID}.cif", cif)
print("타깃 구조", cif, "|", cif.stat().st_size, "bytes")
print("수십 KB 이상이면 정상이고, 몇 백 bytes 면 PDB ID 오타로 오류 페이지를 받은 거예요.")

## 2) 파일 안을 먼저 들여다보기 — 체인과 잔기 번호 (본문 2.4)

받은 파일을 그대로 넣지 않아요. **어떤 체인이 들어 있고 잔기 번호가 어디부터 어디까지인지**를 봐야
`include`·`exclude`·`binding` 에 적을 숫자가 정해집니다.

번호는 두 가지가 나옵니다. 파일에 찍힌 잔기 번호와, 명세가 쓰는 `res_index`(체인의 첫 잔기를 1로 세는 순번)예요.
둘이 다를 수 있으니 나란히 확인합니다.

In [ ]:
import gemmi
st = gemmi.read_structure(str(cif)); st.setup_entities()
model = st[0]
CHAIN_IDS = [ch.name for ch in model]
print("구조", st.name, "| 체인", ", ".join(CHAIN_IDS))
print()
print("체인  폴리머  종류        파일 잔기번호   명세 res_index   서열(앞·뒤)")
for ch in model:
    poly = ch.get_polymer()
    if not len(poly):
        continue
    kind = str(poly.check_polymer_type()).split(".")[-1]
    seq = gemmi.one_letter_code([r.name for r in poly])
    shown = seq if len(seq) <= 30 else seq[:14] + "…" + seq[-14:]
    print(f"  {ch.name:<4}{len(poly):>5}  {kind:<11}"
          f"{poly[0].seqid.num:>6}..{poly[-1].seqid.num:<8}"
          f"{poly[0].label_seq:>4}..{poly[-1].label_seq:<10}{shown}")

## 3) 이 체인들을 명세 문법으로 — entity 5종 (본문 2.2)

긴 체인 두 벌(A·B)이 PD-L1 본체이고, 짧은 체인 두 벌(C·D)은 그 표면에 붙은 채로 결정화된 펩타이드예요.
이걸 명세로 옮기려면 entity 문법이 필요합니다. BoltzGen 이 다루는 entity 는 5종이고,
**만들 것**(`protein`·`dna`·`rna`·`ligand`)과 **붙일 것**(`file`)으로 나뉘어요.

In [ ]:
print("""# 만들 것 — 설계 대상
- protein: { id: B, sequence: 80..140 }      # 80~140aa 중 무작위 길이로 설계
- protein: { id: B, sequence: 120 }          # 고정 120aa
- protein: { id: B, sequence: MKLVAA... }    # 서열 고정(재설계/스캐폴드)

# 붙일 것 — 타깃 entity
- dna: { id: D, sequence: ATGCGT }
- rna: { id: R, sequence: AUGCGU }
- ligand: { id: L, ccd: ATP }                                  # PDB 화학성분 사전 3글자 코드
- ligand: { id: L, smiles: "CC(=O)Oc1ccccc1C(=O)O" }           # 아스피린(CCD 에 없을 때)
- file:   { path: target.cif, include: [ { chain: { id: A } } ] }

# res_index 범위 표기 — include / exclude / binding 공통
  45..55        45번부터 55번까지
  ..10          처음부터 10번까지
  185..         185번부터 끝까지
  10,29,33,40..48   개별 + 범위 혼합""")

## 4) 결합부위를 데이터에서 뽑기 (본문 2.5)

본문이 말한 세 단서 중 가장 확실한 건 **이미 붙어 있는 것**이에요.
7uxq 는 PD-L1(체인 A) 표면에 18잔기 펩타이드(체인 C)가 붙은 상태로 풀린 구조라,
C 로부터 5Å 안에 있는 A 의 잔기가 곧 실험으로 검증된 결합부위입니다.

명세에 적을 값은 `res_index` 쪽이므로, 접촉 잔기를 찾은 뒤 **그 번호가 좌표 있는 잔기를 가리키는지**까지
확인합니다(본문 2.8 체크리스트 "결합부위 잔기 번호").

In [ ]:
TARGET, PROBE, CUTOFF = "A", "C", 5.0          # A = PD-L1 타깃, C = 이미 붙어 있는 펩타이드
chains = {ch.name: ch.get_polymer() for ch in model}
tgt, prb = chains[TARGET], chains[PROBE]

probe_pos = [a.pos for r in prb for a in r]
contact = [r for r in tgt if any(a.pos.dist(p) <= CUTOFF for a in r for p in probe_pos)]

def to_range(nums):
    """연속 구간은 a..b, 단독은 a 로 묶어 BoltzGen 범위 문법 문자열을 만듭니다."""
    out, i = [], 0
    while i < len(nums):
        j = i
        while j + 1 < len(nums) and nums[j + 1] == nums[j] + 1:
            j += 1
        out.append(f"{nums[i]}..{nums[j]}" if j > i else str(nums[i]))
        i = j + 1
    return ",".join(out)

BINDING = to_range(sorted(r.label_seq for r in contact))
print(f"체인 {PROBE} 에서 {CUTOFF:g}Å 안에 있는 체인 {TARGET} 잔기 {len(contact)}개")
print("  파일 잔기번호 ", to_range(sorted(r.seqid.num for r in contact)))
print("  명세 res_index", BINDING)

RESOLVED = {r.label_seq for r in tgt}
used = sorted({int(x) for tok in BINDING.split(",") for x in tok.split("..") if x})
bad = [n for n in used if n not in RESOLVED]
print()
print(f"좌표가 있는 res_index 범위 {min(RESOLVED)}..{max(RESOLVED)}")
print("검증 통과 — 명세에 적을 번호가 모두 실제 잔기를 가리켜요." if not bad
      else f"확인 필요 — 좌표 없는 번호 {bad} 는 빼세요.")

## 5) 뺄 것을 빼고 명세로 쓰기 (본문 2.4)

붙일 자리는 정했으니 이제 **뺄 것**입니다. 2)의 서열 끝을 보면 `...HHHHH` — 정제용 His-태그예요.
생물학적 타깃이 아니고 구조도 흔들리니 `exclude` 로 뺍니다. 끝까지 자르는 거라 `N..` 표기가 딱 맞아요.

남은 함정 둘도 여기서 막습니다. 설계 체인 `id` 는 **파일에 없는 문자**로 골라 충돌을 피하고,
`path` 는 **YAML 파일 위치 기준** 상대경로라 `work/my_spec.yaml` 안에서는 `7uxq.cif` 로 씁니다(본문 2.8).

In [ ]:
tail = list(tgt)
k = len(tail)
while k > 0 and tail[k - 1].name == "HIS":      # C말단 His-태그 구간을 뒤에서부터 찾습니다
    k -= 1
assert k < len(tail), "C말단 His-태그가 없어요 — 뺄 구간을 직접 정해 exclude 에 적으세요."
TAG_FROM = tail[k].label_seq
print(f"C말단 His-태그 {len(tail) - k}잔기 → 파일 번호 {tail[k].seqid.num}.. / res_index {TAG_FROM}.. 를 제거")

DESIGN_ID = next(x for x in "ZYXWV" if x not in CHAIN_IDS)
print(f"설계 체인 id = {DESIGN_ID} (파일이 이미 쓰는 {','.join(CHAIN_IDS)} 와 겹치지 않음)")

spec = f"""entities:
  - protein: {{ id: {DESIGN_ID}, sequence: 80..140 }}       # 설계할 바인더
  - file:
      path: {PDB_ID}.cif
      include:       [ {{ chain: {{ id: {TARGET} }} }} ]
      exclude:       [ {{ chain: {{ id: {TARGET}, res_index: {TAG_FROM}.. }} }} ]
      binding_types: [ {{ chain: {{ id: {TARGET}, binding: "{BINDING}" }} }} ]
      reset_res_index: [ {{ chain: {{ id: {TARGET} }} }} ]
      structure_groups: "all"
"""
MY_SPEC = work / "my_spec.yaml"
MY_SPEC.write_text(spec, encoding="utf-8")
print()
print(spec)

## 6) 서열 칸으로 위상학까지 적기 (본문 2.6)

여기까지가 "타깃에 자유롭게 붙이기"예요. 그런데 설계 단백질의 `sequence` 칸은 길이만 적는 자리가 아니라
**어디에 Cys 를 박고, 고리로 닫고, 어느 구간을 helix 로 둘지**까지 적는 자리입니다.
고리형 펩타이드는 Ch.07 에서 이 표기 그대로 실제 설계를 돌립니다.

In [ ]:
print("""- protein:
    id: B
    sequence: 3C8C6C5C3C1C2     # 디자인 3잔기 + Cys + 8잔기 + Cys + ... = 34잔기, Cys 6개
    cyclic: true                # 머리-꼬리 고리화
    secondary_structure:
        helix: 5..15            # 5~15번을 helix 로
        sheet: 20..28           # 20~28번을 sheet 로
    residue_constraints:
      - { position: 1, allowed: A }         # 1번은 Ala 만
      - { position: 3..5, disallowed: CM }  # 3~5번은 Cys/Met 금지
constraints:
  - bond: { atom1: [B, 4, SG], atom2: [B, 26, SG] }    # Cys4-Cys26 이황화
  - bond: { atom1: [B, 13, SG], atom2: [B, 30, SG] }
  - bond: { atom1: [B, 20, SG], atom2: [B, 32, SG] }""")
print()
print("Cys 6개 → 이황화 3쌍(cystine knot). bond 는 [체인, 잔기번호, 원자이름] 형식이고 SG 는 Cys 의 황 원자예요.")

## 7) 핵산 타깃 — 따로 할 일이 없어요 (본문 2.7)

DNA·RNA 타깃도 4)~5)와 완전히 같은 방식입니다. BoltzGen 이 CIF 안의 잔기 코드를 보고
단백질·DNA·RNA 를 자동 구분하니, 핵산 체인을 그냥 `include` 하면 돼요.

In [ ]:
print("""entities:
  - protein: { id: G, sequence: 40..120 }                # DNA 결합 단백질 설계
  - file:
      path: example/denovo_zinc_finger_against_dna/zf.cif
      include: [ { chain: { id: C1 } }, { chain: { id: B1 } } ]   # DNA 이중가닥 두 체인
      structure_groups: "all"
""")

## 8) 돌리기 전 검증 — `boltzgen check` (본문 2.8)

5)에서 쓴 `work/my_spec.yaml` 이 실제로 통과하는지 확인합니다. 체인 ID·잔기 번호·상대경로 중 하나라도
어긋나면 여기서 바로 오류가 나요. 비교 기준으로 레포 예제(`1g13prot.yaml`)도 같이 검사합니다.
`check` 는 GPU 가 필요 없어 CPU 런타임에서도 돌아갑니다(설치 상세는 Ch.03).

In [ ]:
import shutil, subprocess, sys

if not shutil.which("boltzgen"):
    _run(f'"{sys.executable}" -m pip -q install boltzgen')

SRC = ADV_ROOT / ".boltzgen_src"          # 예제 명세는 pip 패키지에 없고 소스 레포에만 있습니다
if not SRC.exists():
    _run(f'git clone --depth 1 https://github.com/HannesStark/boltzgen.git "{SRC}"')

def check(spec):
    """boltzgen check 를 돌리고 결과를 이 셀 출력으로 가져옵니다."""
    r = subprocess.run(["boltzgen", "check", str(spec)], capture_output=True, text=True)
    print(r.stdout.strip() if r.returncode == 0 else r.stderr.strip()[-700:])

if shutil.which("boltzgen"):
    print("=== 내가 쓴 명세 ===")
    check(MY_SPEC)
    print("=== 레포 예제 ===")
    check(SRC / "example/vanilla_protein/1g13prot.yaml")
    print()
    print("두 명세 모두 sequence 를 80..140 으로 뒀으니, Total designed residues 가 그 범위 안이면 정상이에요.")
    print("매 실행마다 값이 달라지는 것도 정상이고요. 오류가 나면 체인 ID·res_index·path 를 다시 보세요.")
else:
    print("boltzgen 설치가 아직 안 됐어요. 이 셀을 한 번 더 실행하면 설치 완료 후 검증까지 이어집니다.")

## 9) 명세를 3D 로 확인 — PyMOL 불필요 (본문 2.8)

`check` 가 남긴 `my_spec.cif` 를 그대로 띄웁니다. 마우스로 회전·확대하면서
**타깃이 맞는지, 잘라낸 자리가 의도대로인지**를 눈으로 확인하세요(N말단 파랑 → C말단 빨강).

In [ ]:
import importlib.util, pathlib
if importlib.util.find_spec("py3Dmol") is None:
    _run(f'"{sys.executable}" -m pip -q install py3Dmol')
import py3Dmol, gemmi

viz = next((p for p in (pathlib.Path("my_spec.cif"), pathlib.Path("1g13prot.cif")) if p.exists()), None)
assert viz is not None, "시각화 CIF 가 없어요 — 위 8) 셀을 먼저 실행하세요."
print("띄울 구조", viz)

pdb = gemmi.read_structure(str(viz)).make_pdb_string()   # 3Dmol.js 는 PDB 를 가장 안정적으로 읽어요
view = py3Dmol.view(width=760, height=540)
view.addModel(pdb, "pdb")
view.setStyle({"cartoon": {"color": "spectrum"}})
view.setBackgroundColor("white")
view.zoomTo()
view.show()

### 정리

타깃 구조를 받아 체인·잔기 번호를 실측하고, 결합부위를 데이터에서 뽑고, 태그를 잘라내고,
`boltzgen check` 로 통과시킨 `work/my_spec.yaml` 하나가 이 챕터의 결과물이에요.
이제 이 명세를 실제로 돌릴 환경이 필요합니다 — Ch.03 에서 설치와 GPU·CUDA 정합을 맞춥니다.